[Reference](https://medium.com/@Rohan_Dutt/10-genai-paradigms-that-will-redefine-the-modern-data-stack-by-the-end-of-2026-fe92bfb21d3b)

# 1. Self-Optimizing Query Engines

In [1]:
import sqlite3
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from collections import defaultdict

# Simulate a self-optimizing query engine
class QueryOptimizer:
    def __init__(self, db_path):
        self.conn = sqlite3.connect(db_path)
        self.query_log = []
        self.performance_model = RandomForestRegressor()
        self.is_trained = False

    def log_query(self, query, execution_time, rows_scanned):
        """Log each query for pattern analysis"""
        self.query_log.append({
            'query': query,
            'execution_time': execution_time,
            'rows_scanned': rows_scanned,
            'table': self._extract_table(query),
            'columns': self._extract_columns(query)
        })

    def _extract_table(self, query):
        # Simple extraction for demo
        if 'FROM' in query:
            return query.split('FROM')[1].split()[0]
        return None

    def _extract_columns(self, query):
        if 'SELECT' in query:
            cols = query.split('SELECT')[1].split('FROM')[0]
            return [c.strip() for c in cols.split(',')]
        return []

    def train_optimizer(self):
        """Train model to predict which columns need indexing"""
        if len(self.query_log) < 10:
            return  # Need minimum data

        df = pd.DataFrame(self.query_log)

        # Feature engineering: column access frequency
        column_usage = defaultdict(int)
        for cols in df['columns']:
            for col in cols:
                column_usage[col] += 1

        # Create training data: predict if column should be indexed
        X = []
        y = []
        for idx, row in df.iterrows():
            for col in row['columns']:
                features = [
                    column_usage[col],
                    row['execution_time'],
                    row['rows_scanned']
                ]
                X.append(features)
                # Label: 1 if slow query, 0 if fast
                y.append(1 if row['execution_time'] > 0.1 else 0)

        self.performance_model.fit(np.array(X), np.array(y))
        self.is_trained = True

    def auto_create_indexes(self):
        """Automatically create indexes based on predictions"""
        if not self.is_trained:
            self.train_optimizer()

        cursor = self.conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()

        for table in tables:
            table_name = table[0]
            cursor.execute(f"PRAGMA table_info({table_name});")
            columns = [col[1] for col in cursor.fetchall()]

            for col in columns:
                # Predict if this column needs indexing
                features = np.array([[1, 0.5, 1000]])  # Simulated features
                should_index = self.performance_model.predict(features)[0]

                if should_index == 1:
                    try:
                        cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_{col} ON {table_name}({col})")
                        print(f"Auto-created index on {table_name}.{col}")
                    except:
                        pass  # Index may already exist

        self.conn.commit()

# Simulate usage
optimizer = QueryOptimizer(':memory:')
# Simulate logged queries
optimizer.log_query("SELECT user_id, amount FROM transactions", 0.5, 10000)
optimizer.log_query("SELECT user_id FROM transactions WHERE amount > 100", 0.8, 5000)
optimizer.auto_create_indexes()

# 2. Autonomous Data Pipelines

In [2]:
import json
from datetime import datetime
import hashlib

class AutonomousPipeline:
    def __init__(self, config_path):
        with open(config_path) as f:
            self.config = json.load(f)
        self.schema_history = {}
        self.drift_alerts = []

    def capture_schema(self, dataset_name, data_sample):
        """Capture current schema fingerprint"""
        schema = {}
        for col, dtype in data_sample.dtypes.items():
            schema[col] = str(dtype)

        schema_hash = hashlib.md5(str(sorted(schema.items())).encode()).hexdigest()

        if dataset_name not in self.schema_history:
            self.schema_history[dataset_name] = schema_hash
            print(f"Initial schema captured for {dataset_name}")
        elif self.schema_history[dataset_name] != schema_hash:
            self._handle_schema_drift(dataset_name, schema)

        return schema

    def _handle_schema_drift(self, dataset_name, new_schema):
        """Auto-heal pipeline on schema change"""
        self.drift_alerts.append({
            'dataset': dataset_name,
            'timestamp': datetime.now(),
            'action_taken': 'AUTO_REWROTE_TRANSFORMATION'
        })

        # Generate new transformation logic
        transformation_code = self._generate_etl_code(new_schema)

        print(f"DRIFT DETECTED in {dataset_name}. Auto-generated new ETL:")
        print(transformation_code)

        # Write new pipeline code
        with open(f'pipelines/{dataset_name}_pipeline.py', 'w') as f:
            f.write(transformation_code)

    def _generate_etl_code(self, schema):
        """Generate Python ETL code based on schema"""
        columns = list(schema.keys())

        code = f"""
import pandas as pd
def transform_{dataset_name}(df):
    # Auto-generated on {datetime.now()}
    # Handle missing values
    for col in {columns}:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('unknown')
        else:
            df[col] = df[col].fillna(0)

    # Auto-cast types based on schema
    return df.astype({schema})
"""
        return code

    def self_monitor(self, dataset_name, data_sample):
        """Continuous monitoring loop"""
        schema = self.capture_schema(dataset_name, data_sample)

        # Check data quality
        null_percent = data_sample.isnull().mean().mean()
        if null_percent > 0.3:
            print(f"ALERT: High null rate ({null_percent:.2%}) in {dataset_name}")

        return schema

# Simulate usage
import pandas as pd
pipeline = AutonomousPipeline('config.json')

# Initial run
df1 = pd.DataFrame({'user_id': [1,2,3], 'amount': [10.5, 20.1, 15.0]})
pipeline.self_monitor('transactions', df1)

# Schema drift occurs
df2 = pd.DataFrame({'user_id': [1,2,3], 'amount': [10.5, 20.1, 15.0],
                    'new_column': ['A','B','C']})
pipeline.self_monitor('transactions', df2)

# 3. Hyper-Personalized Synthetic Data

In [3]:
import pandas as pd
from sdv.tabular import CTGAN
from sdv.metadata import Metadata
import numpy as np

class SyntheticDataGenerator:
    def __init__(self, real_data: pd.DataFrame):
        self.real_data = real_data
        self.metadata = self._create_metadata()
        self.model = CTGAN(epochs=500)

    def _create_metadata(self):
        """Auto-detect field types and relationships"""
        metadata = Metadata()
        metadata.detect_from_dataframe(self.real_data)

        # Mark sensitive fields for differential privacy
        for col in self.real_data.columns:
            if 'email' in col or 'ssn' in col:
                metadata.update_column(
                    table_name='data',
                    column_name=col,
                    sdtype='email' if 'email' in col else 'ssn'
                )
        return metadata

    def train(self):
        """Train CTGAN on real data"""
        print(f"Training on {len(self.real_data)} rows...")
        self.model.fit(self.real_data)
        print("Model trained. Ready to generate synthetic data.")

    def generate(self, num_rows: int) -> pd.DataFrame:
        """Generate statistically identical synthetic data"""
        synthetic_data = self.model.sample(num_rows)

        # Validate statistical similarity
        self._validate_quality(synthetic_data)

        return synthetic_data

    def _validate_quality(self, synthetic_data):
        """Check column correlation preservation"""
        real_corr = self.real_data.select_dtypes(include=[np.number]).corr()
        synth_corr = synthetic_data.select_dtypes(include=[np.number]).corr()

        correlation_diff = np.abs(real_corr - synth_corr).mean().mean()
        print(f"Average correlation difference: {correlation_diff:.3f}")

        if correlation_diff > 0.15:
            print("WARNING: High statistical divergence detected")
        else:
            print("✓ Statistical quality preserved")

# Real healthcare data (simulated)
real_data = pd.DataFrame({
    'patient_id': [f'P{i:05d}' for i in range(1000)],
    'age': np.random.randint(18, 90, 1000),
    'cholesterol': np.random.normal(200, 40, 1000),
    'treatment_outcome': np.random.choice(['success', 'failure'], 1000),
    'blood_pressure': np.random.normal(120, 15, 1000)
})

# Generate synthetic version
generator = SyntheticDataGenerator(real_data)
generator.train()
synth_data = generator.generate(10000)

# Prove privacy: no real IDs leaked
print(f"Real IDs in synthetic? {any(real_data['patient_id'].isin(synth_data['patient_id']))}")

# 4. Zero-Shot Data Integration

In [4]:
import pandas as pd
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
import json

class ZeroShotIntegrator:
    def __init__(self, llm_model="gpt-4"):
        self.llm = OpenAI(model=llm_model, temperature=0)

    def analyze_schema(self, data_sample: pd.DataFrame, source_name: str):
        """Generate semantic understanding of schema"""
        schema_info = []
        for col in data_sample.columns:
            sample_vals = data_sample[col].dropna().head(3).tolist()
            schema_info.append({
                'column': col,
                'dtype': str(data_sample[col].dtype),
                'sample_values': sample_vals
            })

        prompt = PromptTemplate(
            input_variables=["schema", "source"],
            template="""
            Analyze this schema and identify semantic meaning:
            Source: {source}
            Schema: {schema}

            Return JSON with:
            1. entity_type: What business entity is this? (customer, transaction, product)
            2. primary_key: Likely primary key column
            3. important_metrics: List of numeric business metrics
            4. temporal_cols: Date/time columns
            """
        )

        response = self.llm(prompt.format(
            schema=json.dumps(schema_info),
            source=source_name
        ))

        return json.loads(response)

    def generate_merge_sql(self, source1_df, source2_df, source1_meta, source2_meta):
        """Auto-generate JOIN logic"""
        prompt = PromptTemplate(
            input_variables=["meta1", "meta2"],
            template="""
            Create SQL to merge these two datasets:
            Dataset 1: {meta1}
            Dataset 2: {meta2}

            Provide:
            1. JOIN type (INNER, LEFT, etc.)
            2. JOIN keys with confidence score
            3. Column name mapping if needed
            4. Final SELECT statement

            Return JSON with keys: join_type, join_keys, select_statement
            """
        )

        response = self.llm(prompt.format(
            meta1=json.dumps(source1_meta),
            meta2=json.dumps(source2_meta)
        ))

        return json.loads(response)

    def integrate(self, source1_df, source2_df, source1_name, source2_name):
        """End-to-end zero-shot integration"""
        # Phase 1: Semantic analysis
        meta1 = self.analyze_schema(source1_df, source1_name)
        meta2 = self.analyze_schema(source2_df, source2_name)

        # Phase 2: Generate merge strategy
        merge_plan = self.generate_merge_sql(source1_df, source2_df, meta1, meta2)

        # Phase 3: Execute integration
        # Save to temp tables and run AI-generated SQL
        source1_df.to_csv('temp_source1.csv', index=False)
        source2_df.to_csv('temp_source2.csv', index=False)

        print(f"Integration plan: {merge_plan}")

        # Simulate execution
        merged = pd.merge(
            source1_df,
            source2_df,
            left_on=merge_plan['join_keys']['left'],
            right_on=merge_plan['join_keys']['right'],
            how=merge_plan['join_type'].lower()
        )

        return merged, merge_plan

# Example: Unstructured logs + SQL data
logs_df = pd.DataFrame({
    'event_time': ['2024-01-01 10:00', '2024-01-01 11:00'],
    'user': ['u123', 'u456'],
    'action': ['purchase', 'view']
})

sql_df = pd.DataFrame({
    'id': [123, 456],
    'name': ['Alice', 'Bob'],
    'revenue': [100.0, 0.0]
})

integrator = ZeroShotIntegrator()
result, plan = integrator.integrate(logs_df, sql_df, 'web_logs', 'user_db')

print(f"Successfully integrated {len(result)} rows")
print(f"Join keys: {plan['join_keys']}")

# 5. In-Database ML (No More Extract-Train-Deploy)

In [5]:
import snowflake.connector
from snowflake.ml.registry import Registry
import pandas as pd

class InDatabaseML:
    def __init__(self, account, user, password, database, schema):
        self.conn = snowflake.connector.connect(
            account=account,
            user=user,
            password=password,
            database=database,
            schema=schema
        )
        self.cursor = self.conn.cursor()

    def create_training_view(self):
        """Create feature view directly in Snowflake"""
        sql = """
        CREATE OR REPLACE VIEW customer_features AS
        SELECT
            customer_id,
            COUNT(DISTINCT order_id) as order_count,
            SUM(amount) as total_spend,
            DATEDIFF(day, first_order_date, CURRENT_DATE) as customer_age,
            CASE WHEN churned THEN 1 ELSE 0 END as target
        FROM transactions t
        JOIN customers c ON t.customer_id = c.id
        GROUP BY customer_id, first_order_date, churned
        """
        self.cursor.execute(sql)
        print("Feature view created in Snowflake")

    def train_model_in_db(self, model_name='churn_predictor'):
        """Train XGBoost model inside Snowflake"""
        train_sql = f"""
        CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION {model_name}(
            INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'customer_features'),
            TARGET_COLNAME => 'target',
            CONFIG_OBJECT => {{
                'max_depth': 6,
                'learning_rate': 0.1
            }}
        )
        """
        self.cursor.execute(train_sql)
        print(f"Model {model_name} trained in-database")

        # Register in model registry
        reg = Registry(self.conn)
        reg.add_model(model_name, version_name='v1')

    def batch_predict(self, model_name='churn_predictor'):
        """Score entire table without data export"""
        predict_sql = f"""
        CREATE OR REPLACE TABLE churn_scores AS
        SELECT
            customer_id,
            {model_name}!PREDICT(
                OBJECT_CONSTRUCT(*)
            ) as churn_probability
        FROM customer_features
        """
        self.cursor.execute(predict_sql)

        # Fetch results
        self.cursor.execute("SELECT * FROM churn_scores LIMIT 5")
        results = self.cursor.fetchall()
        return pd.DataFrame(results, columns=['customer_id', 'churn_probability'])

    def monitor_drift(self):
        """Monitor model drift in real-time"""
        drift_sql = """
        SELECT
            MODEL_NAME,
            AVG(ABS(PREDICTION - ACTUAL_LABEL)) as avg_error,
            COUNT(*) as predictions_count
        FROM ML_PREDICTIONS
        WHERE prediction_time > DATEADD(hour, -24, CURRENT_DATE)
        GROUP BY MODEL_NAME
        HAVING avg_error > 0.3
        """
        self.cursor.execute(drift_sql)
        return self.cursor.fetchall()

# Usage
ml = InDatabaseML('account', 'user', 'password', 'ANALYTICS', 'PRODUCTION')
ml.create_training_view()
ml.train_model_in_db()
predictions = ml.batch_predict()

# 6. Conversational Schema Design

In [6]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage
import json

class ConversationalSchemaDesigner:
    def __init__(self, model="gpt-4"):
        self.llm = ChatOpenAI(model=model, temperature=0.1)

    def design_schema(self, requirements: str, tech_stack: list):
        """Convert natural language to DDL"""
        system_prompt = f"""
        You are a senior data architect. Generate production-ready database schemas.
        Follow these rules:
        1. Use appropriate data types for {', '.join(tech_stack)}
        2. Add created_at/updated_at timestamps
        3. Include soft delete columns (is_deleted)
        4. Suggest indexes for common queries
        5. Use UUIDs for distributed systems
        6. Normalize to 3NF unless denormalization is explicitly needed
        """

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"Design a schema for: {requirements}")
        ]

        response = self.llm(messages).content

        # Parse DDL and metadata
        schema = self._parse_response(response)
        return schema

    def _parse_response(self, response):
        """Extract DDL and suggestions from LLM output"""
        lines = response.split('\n')
        ddl_statements = []
        indexes = []
        partitions = []

        for line in lines:
            line = line.strip()
            if line.startswith('CREATE TABLE'):
                ddl_statements.append(line)
            elif 'CREATE INDEX' in line:
                indexes.append(line)
            elif 'PARTITION BY' in line:
                partitions.append(line)

        return {
            'ddl': '\n'.join(ddl_statements),
            'indexes': indexes,
            'partitions': partitions,
            'raw': response
        }

    def generate_migration(self, existing_schema: str, new_requirements: str):
        """Generate ALTER TABLE statements"""
        prompt = f"""
        Existing schema:
        {existing_schema}

        New requirements:
        {new_requirements}

        Generate migration SQL that is backward compatible.
        """

        messages = [
            SystemMessage(content="Generate safe, backward-compatible migrations."),
            HumanMessage(content=prompt)
        ]

        return self.llm(messages).content

    def validate_schema(self, ddl: str, sample_queries: list):
        """Check if schema supports required queries"""
        prompt = f"""
        Schema: {ddl}

        Required queries:
        {chr(10).join(sample_queries)}

        Identify any missing indexes or schema issues.
        """

        messages = [
            SystemMessage(content="You are a schema validator."),
            HumanMessage(content=prompt)
        ]

        return self.llm(messages).content

# Design schema for Shopify + IoT inventory
designer = ConversationalSchemaDesigner()
schema = designer.design_schema(
    requirements="Real-time inventory tracking with Shopify + IoT sensors. Track stock levels, temperature, location, and predict reorder points.",
    tech_stack=["PostgreSQL", "TimescaleDB"]
)

print(schema['ddl'])
print("\nSuggested Indexes:")
for idx in schema['indexes']:
    print(idx)

# 7. AI Data Micro Governance and Surveillance

In [7]:
import re
from presidio_analyzer import AnalyzerEngine, PatternRecognizer
from presidio_anonymizer import AnonymizerEngine
import pandas as pd
from collections import defaultdict

class AIFirstGovernance:
    def __init__(self):
        self.analyzer = AnalyzerEngine()
        self.anonymizer = AnonymizerEngine()
        self.lineage_graph = defaultdict(list)

        # Custom recognizers for business-specific PII
        self._add_custom_recognizers()

    def _add_custom_recognizers(self):
        """Add company-specific sensitive patterns"""
        # Recognize customer_id patterns like CUST_12345
        cust_pattern = PatternRecognizer(
            supported_entity="CUSTOMER_ID",
            name="customer_id_pattern",
            patterns=[{"regex": r"CUST_\d{5,10}", "score": 0.9}]
        )
        self.analyzer.registry.add_recognizer(cust_pattern)

        # Recognize internal project codes
        project_pattern = PatternRecognizer(
            supported_entity="PROJECT_CODE",
            patterns=[{"regex": r"PROJ-[A-Z]{3}-\d{4}", "score": 0.85}]
        )
        self.analyzer.registry.add_recognizer(project_pattern)

    def scan_dataframe(self, df: pd.DataFrame, source: str):
        """Scan entire DataFrame for PII and compliance risks"""
        risk_report = {
            'source': source,
            'pii_columns': [],
            'risk_score': 0,
            'violations': []
        }

        for col in df.columns:
            # Sample for scanning
            sample_text = ' '.join(df[col].astype(str).sample(min(50, len(df))).tolist())

            # Analyze for PII
            results = self.analyzer.analyze(
                text=sample_text,
                language='en'
            )

            if results:
                risk_report['pii_columns'].append({
                    'column': col,
                    'entities': [r.entity_type for r in results],
                    'confidence': np.mean([r.score for r in results])
                })

                # Check GDPR compliance
                if any(r.entity_type in ['EMAIL', 'PHONE_NUMBER', 'CUSTOMER_ID'] for r in results):
                    # Check for consent tracking
                    if 'consent' not in col.lower() and 'opt_in' not in df.columns.astype(str).str.lower().tolist():
                        risk_report['violations'].append(f"Missing consent column for PII in {col}")

        # Calculate overall risk score
        risk_report['risk_score'] = min(100, len(risk_report['pii_columns']) * 20)

        return risk_report

    def track_lineage(self, source_table, transformation_logic, target_table):
        """Auto-track data lineage"""
        self.lineage_graph[target_table].append({
            'source': source_table,
            'transformation': transformation_logic,
            'timestamp': pd.Timestamp.now()
        })

        # Predict impact of schema changes
        if len(self.lineage_graph[target_table]) > 5:
            print(f"WARNING: {target_table} has complex lineage. Schema changes may break {len(self.lineage_graph[target_table])} dependencies.")

    def generate_audit_report(self, table_list, regulation='GDPR'):
        """Generate compliance report for audit"""
        report = {
            'regulation': regulation,
            'tables_assessed': len(table_list),
            'compliant_tables': 0,
            'issues': []
        }

        for table in table_list:
            # Simulate table scan
            risk = self.scan_dataframe(
                pd.DataFrame({'sample': ['data']}),
                source=table
            )

            if risk['risk_score'] < 30 and not risk['violations']:
                report['compliant_tables'] += 1
            else:
                report['issues'].append({
                    'table': table,
                    'risk_score': risk['risk_score'],
                    'violations': risk['violations']
                })

        report['compliance_rate'] = report['compliant_tables'] / report['tables_assessed']
        return report

# Scan production data
governance = AIFirstGovernance()
df = pd.read_csv('customer_data.csv')
risk_report = governance.scan_dataframe(df, source='crm_prod')

print(f"Risk Score: {risk_report['risk_score']}")
print(f"PII Columns Found: {len(risk_report['pii_columns'])}")

# 8. Real-Time Event-Driven Micro-analytics

In [8]:
import asyncio
import json
from datetime import datetime
import numpy as np
from collections import deque
import joblib

class RealtimeMicroanalytics:
    def __init__(self, model_path):
        self.model = joblib.load(model_path)
        self.event_buffer = deque(maxlen=1000)  # Rolling window
        self.anomaly_threshold = 0.95
        self.processing_times = []

    async def process_event(self, event: dict):
        """Process single event with AI inference"""
        start_time = datetime.now()

        # Enrich event with context
        enriched = self._enrich_event(event)

        # Extract features for model
        features = self._extract_features(enriched)

        # Real-time prediction
        prediction = self.model.predict_proba([features])[0]
        risk_score = prediction[1]

        # Take action based on threshold
        if risk_score > self.anomaly_threshold:
            await self._trigger_action(enriched, risk_score)

        # Log for analytics
        self.event_buffer.append({
            'event_id': event['id'],
            'risk_score': risk_score,
            'processed_at': datetime.now()
        })

        processing_time = (datetime.now() - start_time).microseconds / 1000
        self.processing_times.append(processing_time)

        return {
            'event_id': event['id'],
            'risk_score': float(risk_score),
            'latency_ms': processing_time
        }

    def _enrich_event(self, event):
        """Add real-time context"""
        # Add rolling statistics from buffer
        recent_scores = [e['risk_score'] for e in self.event_buffer]

        event['context'] = {
            'avg_risk_last_100': np.mean(recent_scores) if recent_scores else 0,
            'event_velocity': len([e for e in self.event_buffer
                                 if (datetime.now() - e['processed_at']).seconds < 60])
        }
        return event

    def _extract_features(self, event):
        """Feature engineering per event"""
        return [
            event['amount'],
            event['context']['avg_risk_last_100'],
            event['context']['event_velocity'],
            len(event.get('user_agent', '')),
            event.get('ip_country') == 'high_risk'
        ]

    async def _trigger_action(self, event, risk_score):
        """Async action on high-risk event"""
        print(f"🚨 HIGH RISK: {event['id']} score={risk_score:.2f}")
        # In production: call webhook, update KV store, notify user
        await asyncio.sleep(0.001)  # Simulate webhook call

    async def event_stream_processor(self, event_source):
        """Process infinite event stream"""
        tasks = []
        for event in event_source:
            task = asyncio.create_task(self.process_event(event))
            tasks.append(task)

            # Backpressure handling
            if len(tasks) > 100:
                done, tasks = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)

        await asyncio.gather(*tasks)

# Simulate event stream
async def generate_events():
    for i in range(1000):
        yield {
            'id': f'txn_{i}',
            'amount': np.random.exponential(100),
            'user_agent': 'Mozilla',
            'ip_country': 'high_risk' if i % 100 == 0 else 'US'
        }
        await asyncio.sleep(0.001)  # 1000 events/sec

# Run real-time processor
async def main():
    processor = RealtimeMicroanalytics('fraud_model.pkl')
    await processor.event_stream_processor(generate_events())

    avg_latency = np.mean(processor.processing_times)
    print(f"Average latency: {avg_latency:.2f}ms")

asyncio.run(main())

# 9. LLMs as Data Advisors

In [9]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage
import ast
import subprocess

class LLMDataEngineer:
    def __init__(self, model="gpt-4"):
        self.llm = ChatOpenAI(model=model, temperature=0)
        self.code_history = []

    def generate_pipeline(self, requirement: str, source_type: str, target_type: str):
        """Generate complete data pipeline from description"""
        system_prompt = """
        You are a senior data engineer. Generate production-ready Python ETL code.
        Requirements:
        1. Use pandas or pyspark
        2. Include error handling with try/except
        3. Add logging for each step
        4. Include data quality checks
        5. Return JSON with: code, tests, requirements.txt
        """

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"""
            Create a pipeline that: {requirement}
            Source: {source_type}
            Target: {target_type}
            Include idempotency and incremental processing.
            """)
        ]

        response = self.llm(messages).content

        # Parse the response
        parsed = self._extract_code_blocks(response)

        # Auto-generate unit tests
        tests = self._generate_tests(parsed['main_code'])

        self.code_history.append({
            'requirement': requirement,
            'code': parsed,
            'tests': tests
        })

        return {
            'pipeline_code': parsed['main_code'],
            'helper_functions': parsed.get('helper_code', ''),
            'tests': tests,
            'dependencies': parsed.get('requirements', 'pandas\nnumpy')
        }

    def _extract_code_blocks(self, response):
        """Extract Python code from LLM response"""
        blocks = {'main_code': '', 'helper_code': '', 'requirements': ''}

        current_block = None
        for line in response.split('\n'):
            if '```python' in line:
                continue
            elif '```' in line:
                current_block = None
            elif 'requirements.txt' in line.lower():
                current_block = 'requirements'
            elif 'helper' in line.lower():
                current_block = 'helper_code'
            else:
                if current_block:
                    blocks[current_block] += line + '\n'
                else:
                    blocks['main_code'] += line + '\n'

        return blocks

    def _generate_tests(self, code):
        """Generate unit tests for the pipeline"""
        tree = ast.parse(code)
        functions = [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]

        test_code = f"""
import unittest
import pandas as pd

class TestPipeline(unittest.TestCase):
    def setUp(self):
        self.sample_data = pd.DataFrame({{'id': [1,2,3], 'value': [10,20,30]}})

    {'\n    '.join([f'def test_{f}(self): self.fail("Implement test for {f}")' for f in functions])}

if __name__ == '__main__':
    unittest.main()
"""
        return test_code

    def deploy_to_airflow(self, pipeline_code: str, dag_id: str):
        """Convert pipeline to Airflow DAG"""
        dag_template = f"""
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

def pipeline_task():
{chr(10).join(['    ' + line for line in pipeline_code.split(chr(10))])}

with DAG('{dag_id}', start_date=datetime(2024,1,1), schedule='@daily') as dag:
    task = PythonOperator(task_id='etl_task', python_callable=pipeline_task)
"""

        # Validate DAG syntax
        try:
            ast.parse(dag_template)
            print("✓ Airflow DAG generated successfully")
            return dag_template
        except SyntaxError as e:
            print(f"DAG generation failed: {e}")
            return None

# Generate pipeline from natural language
engineer = LLMDataEngineer()
pipeline = engineer.generate_pipeline(
    requirement="Convert CRM contacts + Stripe invoices into a customer LTV model",
    source_type="Salesforce + Stripe API",
    target_type="Snowflake table"
)

# Write to file
with open('generated_pipeline.py', 'w') as f:
    f.write(pipeline['pipeline_code'])

print(f"Generated {len(pipeline['pipeline_code'].split(chr(10)))} lines of code")

# 10. The End of Static Dashboards

In [10]:
import pandas as pd
from langchain.agents import create_pandas_dataframe_agent
from langchain.chat_models import ChatOpenAI
import matplotlib.pyplot as plt
import seaborn as sns

class ConversationalAnalytics:
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.llm = ChatOpenAI(model="gpt-4", temperature=0)
        self.agent = create_pandas_dataframe_agent(
            self.llm,
            self.df,
            verbose=True,
            allow_dangerous_code=True  # In production, sandbox this
        )

    def ask(self, question: str):
        """Ask business question in natural language"""
        # Enhance question with analytical context
        enhanced_prompt = f"""
        As a data analyst, answer: {question}

        Provide:
        1. Direct answer with numbers
        2. Key visualizations (describe them)
        3. Root cause hypotheses
        4. Suggested next analyses

        Use statistical methods where appropriate.
        """

        response = self.agent.run(enhanced_prompt)
        return response

    def generate_executive_summary(self, metric: str, period: str):
        """Auto-generate executive summary"""
        prompt = f"""
        Analyze {metric} for {period} and provide executive summary:
        - YoY change with statistical significance
        - Top 3 contributing factors
        - Actionable recommendations
        Format as markdown with tables.
        """

        return self.agent.run(prompt)

    def create_root_cause_analysis(self, anomaly_date: str, metrics: list):
        """Deep dive into anomalies"""
        # First, extract data around anomaly
        anomaly_data = self.df[
            (self.df['date'] >= pd.Timestamp(anomaly_date) - pd.Timedelta(days=7)) &
            (self.df['date'] <= pd.Timestamp(anomaly_date) + pd.Timedelta(days=7))
        ]

        prompt = f"""
        I see an anomaly on {anomaly_date}.
        Data: {anomaly_data[metrics].to_dict()}

        Perform root cause analysis:
        1. Calculate z-scores for each metric
        2. Identify which metrics deviated first
        3. Check for external factors (holidays, etc.)
        4. Segment analysis (by region, product, etc.)

        Return structured findings.
        """

        return self.agent.run(prompt)

    def auto_visualize(self, question: str):
        """Generate code for visualization"""
        response = self.ask(f"Create Python code to visualize: {question}")

        # Extract code block
        code_block = self._extract_code(response)

        # Execute in sandboxed environment
        try:
            exec(code_block)
            plt.savefig('auto_chart.png')
            print("Visualization saved to auto_chart.png")
        except Exception as e:
            print(f"Visualization failed: {e}")

    def _extract_code(self, response):
        """Extract Python code from response"""
        if '```python' in response:
            return response.split('```python')[1].split('```')[0]
        return response

# Load sales data
sales_df = pd.read_csv('sales_data.csv')
sales_df['date'] = pd.to_datetime(sales_df['date'])

# Conversational interface
analytics = ConversationalAnalytics(sales_df)

# Ask complex business question
answer = analytics.ask("Why did Q3 sales drop 15% compared to Q2?")

# Generate executive summary
summary = analytics.generate_executive_summary('revenue', 'Q3 2024')
print(summary)